# 02 — Anomaly Window Extraction

**Goal:** Detect anomalous windows across the dataset using z-score thresholding on sensor features.

**Method:**
1. Load supervised dataset (6m window, 6.2h label shift)
2. Compute z-scores for ALL windows relative to normal-operation statistics (mean, std from label=0)
3. Flag sensors as "activated" when |z| > 2.0
4. Build binary activation matrix (all windows × sensors)
5. Classify windows: clean-normal, anomalous-normal, clean-pre-failure, anomalous-pre-failure

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(r"C:\Users\Emırhan\Desktop\MASTER THESIS\Kurtulus-thesis")
DATASET_DIR = ROOT / "outputs" / "supervised_datasets" / "6m_22326_shift"

Z_THRESH = 2.0

## 1. Load Dataset

In [2]:
train = pd.read_parquet(DATASET_DIR / "train.parquet")
test = pd.read_parquet(DATASET_DIR / "test.parquet")

exclude = {"window_end", "label", "days_since_last_failure", "days_since_last_revision"}
feature_cols = [c for c in train.columns if c not in exclude]

suffixes = ("_min", "_max", "_mean", "_sum", "_active_s", "_flips")
sensors = sorted({col[: -len(suf)] for col in feature_cols for suf in suffixes if col.endswith(suf)})

normal = train[train["label"] == 0]
pre_failure = train[train["label"] == 1]

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")
print(f"Feature columns: {len(feature_cols)}")
print(f"Base sensors: {len(sensors)}")
print(f"Normal windows: {len(normal)}")
print(f"Pre-failure windows (label=1): {len(pre_failure)}")

Train: (40517, 348)
Test: (21554, 348)
Feature columns: 344
Base sensors: 98
Normal windows: 39999
Pre-failure windows (label=1): 518


## 2. Compute Z-Scores for ALL Windows

Z-score each feature relative to normal-operation statistics (mean, std computed from label=0 windows only).

In [3]:
# Compute normal statistics
normal_stats = {}
for col in feature_cols:
    mu = normal[col].mean()
    sd = normal[col].std()
    normal_stats[col] = {"mean": float(mu), "std": float(sd)}

# Z-score ALL windows
z_data = {}
for col in feature_cols:
    mu = normal_stats[col]["mean"]
    sd = normal_stats[col]["std"]
    if sd < 1e-10:
        z_data[col] = np.zeros(len(train))
    else:
        z_data[col] = ((train[col] - mu) / sd).values

z_scores = pd.DataFrame(z_data, index=train.index)
print(f"Z-score matrix: {z_scores.shape}")

Z-score matrix: (40517, 344)


## 3. Build Sensor Activation Matrix

A sensor is "activated" in a window if ANY of its features (min, max, mean, sum, active_s, flips) has |z-score| > 2.0.

In [4]:
# Build binary activation matrix: all windows x sensors
activation = pd.DataFrame(0, index=train.index, columns=sensors)

for sensor in sensors:
    sensor_feats = [c for c in feature_cols if any(c == sensor + suf for suf in suffixes)]
    if sensor_feats:
        max_abs_z = z_scores[sensor_feats].abs().max(axis=1)
        activation[sensor] = (max_abs_z > Z_THRESH).astype(int)

print(f"Activation matrix: {activation.shape}")
print(f"Total sensor activations: {activation.sum().sum()}")
print(f"Avg sensors activated per window: {activation.sum(axis=1).mean():.1f}")

Activation matrix: (40517, 98)
Total sensor activations: 412242
Avg sensors activated per window: 10.2


## 4. Window Classification

Categorize each window into 4 groups based on (has anomaly?) x (label).

In [5]:
n_activated_per_window = activation.sum(axis=1)
is_anomalous = n_activated_per_window > 0
labels = train["label"].values

n_total = len(train)
n_any_anomaly = is_anomalous.sum()
n_normal_clean = int(((~is_anomalous) & (labels == 0)).sum())
n_normal_anomalous = int(((is_anomalous) & (labels == 0)).sum())
n_prefailure_clean = int(((~is_anomalous) & (labels == 1)).sum())
n_prefailure_anomalous = int(((is_anomalous) & (labels == 1)).sum())

classification = pd.DataFrame({
    "Category": [
        "Normal + No Activation (clean normal)",
        "Normal + Has Activation (anomalous, no failure)",
        "Pre-failure + No Activation (failure without deviation)",
        "Pre-failure + Has Activation (anomalous + failure)",
    ],
    "Count": [n_normal_clean, n_normal_anomalous, n_prefailure_clean, n_prefailure_anomalous],
    "% of Total": [
        f"{100*n_normal_clean/n_total:.1f}%",
        f"{100*n_normal_anomalous/n_total:.1f}%",
        f"{100*n_prefailure_clean/n_total:.1f}%",
        f"{100*n_prefailure_anomalous/n_total:.1f}%",
    ],
})

print(f"Total windows: {n_total}")
print(f"Windows with >= 1 sensor activation: {n_any_anomaly} ({100*n_any_anomaly/n_total:.1f}%)\n")
classification

Total windows: 40517
Windows with >= 1 sensor activation: 28663 (70.7%)



,Category,Count,% of Total
0,Normal + No Activation (clean normal),11834,29.2%
1,"Normal + Has Activation (anomalous, no failure)",28165,69.5%
2,Pre-failure + No Activation (failure without d...,20,0.0%
3,Pre-failure + Has Activation (anomalous + fail...,498,1.2%
